# Transcrição de imagens com OpenRouter
Abra este notebook no Google Colab e execute as células em ordem. A chave é solicitada em um campo oculto e não fica gravada no notebook.

O notebook baixa o ZIP do Drive, extrai as imagens da pasta `Imagens` e faz uma requisição por imagem, em ordem numérica. O arquivo JSON de saída registra o modelo escolhido e, para cada imagem, a transcrição e o custo informado pelo OpenRouter. O DOCX existente no ZIP não é utilizado.

O modelo e o esforço padrão permanecem configurados nas variáveis `MODEL` e `REASONING_EFFORT`. Para trocar de modelo, altere `MODEL` para um identificador disponível no OpenRouter. A compatibilidade com esforço de raciocínio varia conforme o modelo.

O arquivo do Drive precisa permitir download pela conta utilizada; para download direto com `gdown`, compartilhe-o como “Qualquer pessoa com o link”.


In [ ]:
%pip -q install gdown requests

In [ ]:
import gdown
import requests
import base64
import mimetypes
import json
import re
import tempfile
import zipfile

from getpass import getpass
from pathlib import Path
from google.colab import userdata


API_KEY = userdata.get("API_KEY")
if not API_KEY:
    raise ValueError("Informe sua chave do OpenRouter.")

MODEL = "google/gemini-3.1-pro-preview"
REASONING_EFFORT = "high"
DRIVE_URL = "https://drive.google.com/file/d/1uEv67wMkSAY_31YyizjPSXufZAy2Dndp/view?usp=drive_link"


def conversar(conteudo, model=MODEL):
    resposta = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": conteudo}],
            "reasoning": {"effort": REASONING_EFFORT},
            "max_tokens": 8192,
        },
        timeout=(30, 300),
    )

    if not resposta.ok:
        raise RuntimeError(f"OpenRouter: HTTP {resposta.status_code} — {resposta.text}")

    dados = resposta.json()
    if "error" in dados:
        raise RuntimeError(f"OpenRouter: {dados['error']}")

    texto = dados["choices"][0]["message"].get("content")
    if not isinstance(texto, str) or not texto.strip():
        raise RuntimeError("O modelo não retornou texto. Verifique a resposta e o limite de tokens.")

    custo = dados.get("usage", {}).get("cost")
    return texto, custo

def imprimir_custo(custo):
    if custo is not None:
        print(f"\nCusto da requisição: {custo:.6f} créditos OpenRouter")
    else:
        print("\nCusto não informado pela API.")

## Teste opcional de conexão
Execute a célula abaixo somente se quiser enviar “oi” e imprimir a resposta e o custo. Esse teste faz uma requisição adicional.


In [ ]:
resposta_teste, custo_teste = conversar("oi")
print(resposta_teste)
imprimir_custo(custo_teste)

Oi! Como posso te ajudar hoje?

Custo da requisição: 0.002378 créditos OpenRouter


## Baixar o ZIP, extrair e localizar as imagens
As imagens são ordenadas numericamente, por exemplo `5.BR...` antes de `10.BR...`.


In [ ]:
# Uma pasta nova a cada execução evita selecionar arquivos de downloads anteriores.
PASTA = Path(tempfile.mkdtemp(prefix="transcricao_"))
ARQUIVO_ZIP = PASTA / "documentos.zip"
DESTINO = PASTA / "extraido"
DESTINO.mkdir()

def download_drive_file(url, caminho_destino):
    # Extrai o ID do arquivo da URL do Google Drive
    match = re.search(r'[-\w]{25,}', url)
    if not match:
        raise ValueError("Não foi possível extrair o ID do arquivo da URL do Drive.")
    file_id = match.group(0)

    session = requests.Session()
    download_url = "https://docs.google.com/uc?export=download"
    params = {'id': file_id, 'confirm': 't'}

    resposta = session.get(download_url, params=params, stream=True)

    # Caso precise passar pela página de confirmação de arquivos grandes
    token = None
    for key, value in resposta.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break

    if token:
        params['confirm'] = token
        resposta = session.get(download_url, params=params, stream=True)

    resposta.raise_for_status()

    with open(caminho_destino, 'wb') as f:
        for chunk in resposta.iter_content(chunk_size=32768):
            if chunk:
                f.write(chunk)

try:
    print("Iniciando download do arquivo...")
    download_drive_file(DRIVE_URL, ARQUIVO_ZIP)
    print("Download concluído com sucesso.")
except Exception as e:
    raise RuntimeError(f"Erro ao baixar o arquivo: {e}")

if not zipfile.is_zipfile(ARQUIVO_ZIP):
    raise RuntimeError("O arquivo baixado não é um ZIP válido. Verifique o link no Drive.")

with zipfile.ZipFile(ARQUIVO_ZIP) as arquivo:
    for item in arquivo.infolist():
        alvo = (DESTINO / item.filename).resolve()
        if not alvo.is_relative_to(DESTINO.resolve()):
            raise ValueError("O ZIP contém um caminho de extração inválido.")
    arquivo.extractall(DESTINO)


def ordem_natural(caminho):
    return [int(parte) if parte.isdigit() else parte.casefold()
            for parte in re.split(r"(\d+)", caminho.as_posix())]


imagens = sorted(
    (p for p in DESTINO.rglob("*")
     if p.is_file()
     and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
     and any(parte.casefold() == "imagens" for parte in p.relative_to(DESTINO).parts[:-1])
     and "__MACOSX" not in p.parts
     and not p.name.startswith(".")),
    key=ordem_natural,
)
if not imagens:
    raise FileNotFoundError("Nenhuma imagem encontrada na pasta Imagens do ZIP.")

print(f"{len(imagens)} imagem(ns) encontrada(s).")

Iniciando download do arquivo...
Download concluído com sucesso.
12 imagem(ns) encontrada(s).


## Transcrever todas as imagens e salvar o JSON


In [ ]:
ARQUIVO_RESULTADOS = PASTA / "resultados_transcricao.json"
resultados = {
    "modelo": MODEL,
    "esforco_raciocinio": REASONING_EFFORT,
    "imagens": [],
}

for indice, caminho_imagem in enumerate(imagens, start=1):
    print(f"[{indice}/{len(imagens)}] Processando {caminho_imagem.name}")
    mime = mimetypes.guess_type(caminho_imagem.name)[0]
    if mime not in {"image/jpeg", "image/png", "image/webp"}:
        raise ValueError(f"Formato de imagem não suportado: {caminho_imagem.name}")
    imagem_base64 = base64.b64encode(caminho_imagem.read_bytes()).decode("ascii")

    texto, custo = conversar([
        {
            "type": "text",
            "text": (
                "Transcreva todo o texto presente nesta imagem, preservando a grafia "
                "original. Marque trechos ilegíveis como [ilegível] e não utilize quebras de "
                "linha como \n."
                "Retorne apenas a transcrição, sem introdução, explicações ou avaliação."
            ),
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:{mime};base64,{imagem_base64}"},
        },
    ], model=MODEL)

    resultados["imagens"].append({
        "arquivo": caminho_imagem.name,
        "texto_transcrito": texto,
        "custo": custo,
    })
    # Salva após cada resposta para preservar o progresso se uma requisição falhar.
    ARQUIVO_RESULTADOS.write_text(
        json.dumps(resultados, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    imprimir_custo(custo)

print(f"\nResultados salvos em: {ARQUIVO_RESULTADOS}")

[1/12] Processando 5.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1894.jpg

Custo da requisição: 0.059874 créditos OpenRouter
[2/12] Processando 6.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg

Custo da requisição: 0.055074 créditos OpenRouter
[3/12] Processando 7.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg

Custo da requisição: 0.069630 créditos OpenRouter
[4/12] Processando 8.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg

Custo da requisição: 0.057410 créditos OpenRouter
[5/12] Processando 9.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg

Custo da requisição: 0.100494 créditos OpenRouter
[6/12] Processando 10.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg

Custo da requisição: 0.047246 créditos OpenRouter
[7/12] Processando 11.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg

Custo da requisição: 0.064818 créditos OpenRout

## Avaliação das Transcrições

In [ ]:
!pip install jiwer

In [ ]:
import jiwer
import re

In [ ]:
_PUNCT_CHAR = r"(?!(?<=\w)['’-](?=\w))[^\w\s]"
_PUNCT_SUB = re.compile(f"({_PUNCT_CHAR})")
_IS_PUNCT = re.compile(r"^[^\w\s]$")
PUNCT = "[PUNCT]"


def normalize_text(text):
    text = text.lower()
    text = re.sub(_PUNCT_CHAR, "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_separate_punctuation(text):
    text = text.lower()
    text = _PUNCT_SUB.sub(r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
def _mask(tokens):
  return [PUNCT if _IS_PUNCT.match(t) else t for t in tokens]


def _align_punctuation(ref, hyp):
  """Alinha as sequências mascaradas e devolve os pares (i, j) de
  pontuações alinhadas entre si."""
  ref_m, hyp_m = _mask(ref), _mask(hyp)
  n, m = len(ref), len(hyp)
  big = n + m + 1

  def step(i, j):
    if ref_m[i] != hyp_m[j]:
      return big
    if ref_m[i] == PUNCT and ref[i] != hyp[j]:
      return 1
    return 0

  dist = [[0] * (m + 1) for _ in range(n + 1)]
  for i in range(1, n + 1):
    dist[i][0] = i * big
  for j in range(1, m + 1):
    dist[0][j] = j * big
  for i in range(1, n + 1):
    for j in range(1, m + 1):
        dist[i][j] = min(
          dist[i - 1][j - 1] + step(i - 1, j - 1),
          dist[i - 1][j] + big,
          dist[i][j - 1] + big,
        )

  pairs, i, j = [], n, m
  while i > 0 and j > 0:
    if dist[i][j] == dist[i - 1][j - 1] + step(i - 1, j - 1):
      if ref_m[i - 1] == PUNCT and hyp_m[j - 1] == PUNCT:
          pairs.append((i - 1, j - 1))
      i, j = i - 1, j - 1
    elif dist[i][j] == dist[i - 1][j] + big:
      i -= 1
    else:
      j -= 1
  return pairs


def punctuation_counts(reference, hypothesis):
  """Espera textos já passados por normalize_separate_punctuation."""
  ref, hyp = reference.split(), hypothesis.split()
  pairs = _align_punctuation(ref, hyp)

  n_ref = sum(1 for t in ref if _IS_PUNCT.match(t))
  n_hyp = sum(1 for t in hyp if _IS_PUNCT.match(t))
  correct = sum(ref[i] == hyp[j] for i, j in pairs)
  return {
    "C": correct,
    "S": len(pairs) - correct,
    "D": n_ref - len(pairs),
    "I": n_hyp - len(pairs),
  }


def _per(c):
  errors = c["S"] + c["D"] + c["I"]
  total = errors + c["C"]
  return errors / total if total else 0.0


def punctuation_error_rate(reference, hypothesis):
  """
  Cálculo da Punctuation Error Rate (PER), com base em https://arxiv.org/abs/2310.02943

  PER = Ip + Dp + Sp / Ip + Dp + Sp + Cp
  Ip = Pontuações Inseridas
  Dp = Pontuações Deletadas
  Sp = Pontuações Substituídas
  Cp = Pontuações Corretas
  """
  reference = normalize_separate_punctuation(reference)
  hypothesis = normalize_separate_punctuation(hypothesis)
  return _per(punctuation_counts(reference, hypothesis))


def corpus_punctuation_error_rate(references, hypotheses):
  """ PER de um corpus todo"""
  total = {"C": 0, "S": 0, "D": 0, "I": 0}
  for r, h in zip(references, hypotheses):
    for k, v in punctuation_counts(r, h).items():
      total[k] += v
  return _per(total)

In [ ]:
import csv
import json
import unicodedata
from pathlib import Path

ARQUIVO_GABARITOS = "dicionario (1).json"
ARQUIVO_RESULTADOS = "resultados_transcricao (2).json"


def normalize_filename(filename):
    # Os caminhos nos JSONs podem ter pastas diferentes ou Unicode composto/decomposto.
    return unicodedata.normalize("NFC", Path(filename).name)


with open(ARQUIVO_GABARITOS, encoding="utf-8") as f:
    gabaritos = json.load(f)

with open(ARQUIVO_RESULTADOS, encoding="utf-8") as f:
    resultados = json.load(f)

gabaritos_por_arquivo = {normalize_filename(nome): texto for nome, texto in gabaritos.items()}
linhas_csv = []

for imagem in resultados["imagens"]:
    nome_arquivo = normalize_filename(imagem["arquivo"])
    referencia = gabaritos_por_arquivo.get(nome_arquivo)
    transcricao = imagem["texto_transcrito"]

    linha = {
        "arquivo": nome_arquivo,
        "modelo": resultados.get("modelo", ""),
        "esforco_raciocinio": resultados.get("esforco_raciocinio", ""),
        "custo": imagem.get("custo", ""),
    }

    if referencia is None:
        print(f"\n{nome_arquivo}: gabarito não encontrado")
        linha["status"] = "gabarito não encontrado"
        for metrica in ("wer_sem_pontuacao", "cer_sem_pontuacao", "wer_com_pontuacao", "cer_com_pontuacao", "per"):
            linha[metrica] = ""
        linhas_csv.append(linha)
        continue

    ref_sem_pontuacao = normalize_text(referencia)
    hyp_sem_pontuacao = normalize_text(transcricao)
    ref_com_pontuacao = normalize_separate_punctuation(referencia)
    hyp_com_pontuacao = normalize_separate_punctuation(transcricao)

    metricas = {
        "wer_sem_pontuacao": jiwer.wer(ref_sem_pontuacao, hyp_sem_pontuacao),
        "cer_sem_pontuacao": jiwer.cer(ref_sem_pontuacao, hyp_sem_pontuacao),
        "wer_com_pontuacao": jiwer.wer(ref_com_pontuacao, hyp_com_pontuacao),
        "cer_com_pontuacao": jiwer.cer(ref_com_pontuacao, hyp_com_pontuacao),
        "per": punctuation_error_rate(ref_com_pontuacao, hyp_com_pontuacao),
    }
    linha.update(metricas)
    linha["status"] = "ok"
    linhas_csv.append(linha)

    print(f"\nImagem: {nome_arquivo}")
    print(f"WER (sem pontuação): {metricas['wer_sem_pontuacao']:.4f}")
    print(f"CER (sem pontuação): {metricas['cer_sem_pontuacao']:.4f}")
    print(f"WER (com pontuação): {metricas['wer_com_pontuacao']:.4f}")
    print(f"CER (com pontuação): {metricas['cer_com_pontuacao']:.4f}")
    print(f"PER: {metricas['per']:.4f}")

# O nome do CSV acompanha o arquivo JSON de resultados, preservando cada conjunto.
arquivo_csv = Path(ARQUIVO_RESULTADOS).with_name(
    f"{Path(ARQUIVO_RESULTADOS).stem}_metricas.csv"
)
colunas = [
    "arquivo", "modelo", "esforco_raciocinio", "custo", "status",
    "wer_sem_pontuacao", "cer_sem_pontuacao",
    "wer_com_pontuacao", "cer_com_pontuacao", "per",
]
with open(arquivo_csv, "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=colunas)
    escritor.writeheader()
    escritor.writerows(linhas_csv)

print(f"\nResultados salvos em: {arquivo_csv}")
